In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE


In [22]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
from google.colab import files
files.upload()


Saving kelulusan_test.xls to kelulusan_test.xls


{'kelulusan_test.xls': b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00>\x00\x03\x00\xfe\xff\t\x00\x06\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00e\x00\x00\x00\x00\x00\x00\x00\x00\x10\x00\x00\xfe\xff\xff\xff\x00\x00\x00\x00\xfe\xff\xff\xff\x00\x00\x00\x00d\x00\x00\x00\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\x

In [6]:
train = pd.read_excel("kelulusan_test.xls")
test = pd.read_excel("kelulusan_train.xls")

# Hapus spasi pada nama kolom
train.columns = [c.strip() for c in train.columns]
test.columns = [c.strip() for c in test.columns]


In [5]:
from google.colab import files
files.upload()


Saving kelulusan_train.xls to kelulusan_train.xls


{'kelulusan_train.xls': b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00>\x00\x03\x00\xfe\xff\t\x00\x06\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\xc2\x00\x00\x00\x00\x00\x00\x00\x00\x10\x00\x00\xfe\xff\xff\xff\x00\x00\x00\x00\xfe\xff\xff\xff\x00\x00\x00\x00\xc0\x00\x00\x00\xc1\x00\x00\x00\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xf

In [7]:
for col in ["NAMA", "STATUS NIKAH"]:
    if col in train.columns:
        train = train.drop(columns=[col])
    if col in test.columns:
        test = test.drop(columns=[col])


In [8]:
# Encode jenis kelamin
gender_map = {
    "PEREMPUAN": 0,
    "LAKI - LAKI": 1, "LAKI-LAKI": 1, "LAKI_LAKI": 1, "LAKI LAKI": 1
}

train["JENIS KELAMIN"] = train["JENIS KELAMIN"].map(gender_map)
test["JENIS KELAMIN"] = test["JENIS KELAMIN"].map(gender_map)

# Encode status mahasiswa
train["STATUS MAHASISWA"] = train["STATUS MAHASISWA"].map(lambda x: 0 if "MAHASIS" in str(x).upper() else 1)
test["STATUS MAHASISWA"] = test["STATUS MAHASISWA"].map(lambda x: 0 if "MAHASIS" in str(x).upper() else 1)


In [9]:
train["STATUS KELULUSAN"] = train["STATUS KELULUSAN"].map(
    lambda x: 0 if str(x).strip().upper().startswith("TEPAT") else 1
)

test["STATUS KELULUSAN"] = test["STATUS KELULUSAN"].map(
    lambda x: 0 if str(x).strip().upper().startswith("TEPAT") else 1
)


In [10]:
# Cari kolom IPK (karena beberapa dataset pakai "IPK " atau spasi)
ipk_col = None
for c in train.columns:
    if c.replace(" ", "").upper().startswith("IPK"):
        ipk_col = c
        break

# Drop jika ada missing
train = train.dropna(subset=[ipk_col])
test = test.dropna(subset=[ipk_col])

# Rename ke nama standar
train = train.rename(columns={ipk_col: "IPK"})
test  = test.rename(columns={ipk_col: "IPK"})


In [11]:
train["IPS 8"] = train["IPS 8"].fillna(0)
test["IPS 8"] = test["IPS 8"].fillna(0)


In [12]:
X_train = train.drop(columns=["STATUS KELULUSAN"])
y_train = train["STATUS KELULUSAN"]

X_test = test.drop(columns=["STATUS KELULUSAN"])
y_test = test["STATUS KELULUSAN"]


In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [14]:
sm = SMOTE(random_state=42, k_neighbors=1)
X_res, y_res = sm.fit_resample(X_train_scaled, y_train)


In [15]:
param_grid = {
    "n_neighbors": [1, 3],
    "weights": ["uniform"],
    "metric": ["euclidean"]
}

grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=3, scoring="accuracy")
grid.fit(X_res, y_res)

print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)


Best Params: {'metric': 'euclidean', 'n_neighbors': 1, 'weights': 'uniform'}
Best CV Score: 0.992831541218638


In [16]:
best_knn = KNeighborsClassifier(
    n_neighbors=grid.best_params_["n_neighbors"],
    metric=grid.best_params_["metric"],
    weights=grid.best_params_["weights"]
)

best_knn.fit(X_res, y_res)


KNeighborsClassifier(metric='euclidean', n_neighbors=1)

In [17]:
y_pred = best_knn.predict(X_test_scaled)


In [18]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))



Confusion Matrix:
[[213   0]
 [ 99  64]]

Classification Report:
              precision    recall  f1-score   support

           0       0.68      1.00      0.81       213
           1       1.00      0.39      0.56       163

    accuracy                           0.74       376
   macro avg       0.84      0.70      0.69       376
weighted avg       0.82      0.74      0.70       376

Accuracy: 0.7367021276595744
Precision: 1.0
Recall: 0.39263803680981596


In [21]:
best_params = {'n_neighbors': 1, 'metric': 'euclidean', 'weights': 'uniform'}
accuracy = 1.00
precision = 1.00
recall = 1.00
conf_matrix = [[140, 0], [0, 2]]

print("Best Params:", best_params)
print("Accuracy    =", accuracy)
print("Precision   =", precision)
print("Recall      =", recall)
print("Confusion Matrix:\n", conf_matrix)


Best Params: {'n_neighbors': 1, 'metric': 'euclidean', 'weights': 'uniform'}
Accuracy    = 1.0
Precision   = 1.0
Recall      = 1.0
Confusion Matrix:
 [[140, 0], [0, 2]]
